### Multidimensional GP



In [4]:
import os

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.simplefilter('ignore')
# import ipywidgets as widgets
from IPython.display import display
import pickle as pkl
# from sklearn.model_selection import train_test_split
# import plotly.graph_objects as go
import pickle as pkl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## A two dimentional problem

### You are managing a software company and in charged of the server capacity.
### You have different regions (where each region differs by the size of activity) in which you need to supply servers for 24 hours a day. The demand for servers varies with time and region size.

### You have the ability to purchase servers from AWS, however, you need to do it before the demand reveals.

### The decision is need to be made arbitrary over time and for different region.

### For example, the hour is now 05:00 AM, and you need to assign servers for a region which has 20K customers. What is the server capacity the you need to purchase?

### Every capacity unit cost 10 Dollars. However, the damage for not being able to supply a single demand cost 30.

### You have previuos data, which reveals past demand at differt hour of the day and region size.

### Based on the data, what should be the optimal assignment strategy?

### We first try to estimate the demand the best way we can.

### Then, we take a simple approach, we assign supply accirding to the mean value of our prediction, which would work decently.

### We'll also try a linear regression and compare both of them.

## Procedure:

1. Estimating the demand function.
2. Computing the cost function

## Ground Truth

The true demand function looks like this

In [9]:
0.5**5, (2/3)**5

(0.03125, 0.13168724279835387)

In [5]:
colab = False
if colab:
    data, X_data = pkl.load(open('/content/drive/My Drive/Bayesian_programming/data/data_demand.pkl', 'rb'))
else:
    data, X_data = pkl.load(open('../data/data_demand.pkl', 'rb'))

FileNotFoundError: [Errno 2] No such file or directory: '../data/data_demand.pkl'

In [8]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)



# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color='red',             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)
fig = go.Figure(data=[scatter_demand, surface_demand])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=800,  # Adjust width here
    height=800
)

# Show the plot
fig.show()

## Defining a pymc GP model:
### Lets do it buttom to up

### 1. Sampling the posterior
### 2. Defining the likelihood: using gp.marginal_likelihood
### 3. Defining prior to the noise parameter
### 4. Defining the Pymc gp object: this includes both mean function and covariance function
### 5. defining the mean function
### 6. defining the covariance function
### 7. defining prior of the cov function

## Fitting a GP

In [ ]:
with pm.Model() as spatial_model_1:

     # 7 defining prior of the cov function


    # 6 defining the covariance function


    # 5 defining the mean function


    # 4 Defining the Pymc gp object: this includes both mean function and covariance function
    gp_spatial =

    # 3 Defining prior to the noise parameter



    # 2 Defining the likelihood: using gp.marginal_likelihood


    # 1 Sampling the posterior
    mp_res_alloc = pm.find_MAP()



### Sampling a single realization from the posterior

In [ ]:
nd = 30
z1a, z2a = np.meshgrid(np.linspace(0, 24, nd), np.linspace(1, 50, nd))
Z1 = np.concatenate([z1a.reshape(nd*nd, 1), z2a.reshape(nd*nd, 1)], 1)
Z1.shape

(900, 2)

## Lets find the GP prediciton using the .predict function:

In [ ]:

with spatial_model_1:
  mu1, cov1 = gp_spatial.predict(Z1, point=mp_res_alloc)


## Visualizing the results

In [ ]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)

# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface Ground truth'
)

surface_demand_gp = go.Surface(
    x=z1a, y=z2a, z=mu1.reshape(30,30),
    colorscale='plasma',
    opacity=0.7,
    name='Surface GP'
)


fig = go.Figure(data=[scatter_demand, surface_demand, surface_demand_gp])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=700,  # Adjust width here
    height=700,
    showlegend=True
)

# Show the plot
fig.show()

## Linear regression

Lets try do it also with Linear regression

In [ ]:
X_data[:,0]

In [ ]:
with pm.Model() as linear_assignment:

    β = pm.Normal('β', mu=0, sigma=50, shape=3)
    σ = pm.HalfNormal('σ', sigma=50)

    μ = β[0] + β[1] * X_data[:,0] + β[2] * X_data[:,1]

    recruits = pm.Normal('recruits', mu=μ, sigma=σ, observed=data)

    quad_trace = pm.sample(1000, tune=500, chains = 2, cores=2, random_seed=1)

### Aranging the predicition for visualization in a surface:

In [ ]:
samples_linear = quad_trace.posterior['β']
betas = np.array(samples_linear).reshape(-1, 1000, 3)[0,:,:].mean(axis = 0)

lin_reg = []
for x_val in z1a[0,:]:
    for y_val in z2a[:,0]:
        lin_reg.append(betas[0] + betas[1]*x_val + betas[2]*y_val)

lin_reg = np.array(lin_reg).reshape(30,30)

## Visualizing the results

In [ ]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)



# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)

surface_demand_gp = go.Surface(
    x=z1a, y=z2a, z=mu1.reshape(30,30),
    colorscale='plasma',
    opacity=0.7,
    name='Surface'
)
surface_demand_lin = go.Surface(
    x=z1a, y=z2a, z=lin_reg,
    colorscale='plasma',
    opacity=0.7,
    name='Surface'
)


fig = go.Figure(data=[scatter_demand, surface_demand, surface_demand_gp, surface_demand_lin])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=700,  # Adjust width here
    height=700
)

# Show the plot
fig.show()

## Cost function

### Lets follow the assignemt strategy suggested above and compare the total cost based on both linear and gp regressors

In [ ]:
def cost_func(demand, supply, hour, size,  a1 = 10, a2 = 30):

    return  a1*supply + a2*max(0, demand-supply)

def lin_demand(betas, hour, size):

   return betas[0] + betas[1]*hour + betas[2]*size


def mean_gp_val(hour, size):
    with spatial_model_1:
        mu1, cov1 = gp_spatial.predict(np.array([[hour,size]]), point=mp_res_alloc)
    return mu1[0]

def demand_true(hour, size):

    return hour * np.sin(hour / 1.5) * size ** 0.2 + 120

In [ ]:
gp_tot_cost = []
lin_tot_cost = []

for ind in tqdm(range(50)):

  hour = np.random.uniform(0,24)
  size = np.random.uniform(0,50)

  supplygp = mean_gp_val(hour, size)

  demand = demand_true(hour, size)
  supplylin = lin_demand(betas, hour,size)

  cost_lin = cost_func(demand, supplylin, hour, size)
  cost_gp = cost_func(demand, supplygp, hour, size)

  gp_tot_cost.append(cost_gp)
  lin_tot_cost.append(cost_lin)


print('linear cost: ', np.array(lin_tot_cost).mean() )
print('gp cost: ', np.array(gp_tot_cost).mean())
print('Percetage saved: ', 100*(np.array(lin_tot_cost).mean()-np.array(gp_tot_cost).mean())/np.array(lin_tot_cost).mean())

100%|██████████| 50/50 [00:49<00:00,  1.00it/s]

linear cost:  1430.6187086832983
gp cost:  1239.3194539981953
Percetage saved:  13.37178477563526
